In [ ]:
#Notebook description

# this notebook is intended to the utiltities sector

In [ ]:

#Load libraries 
import logging
logger = logging.getLogger('yfinance')
logger.disabled = True
logger.propagate = False
import sys
sys.path.append(r"e:\Coding Projects\Investment Analysis")

# Load libraries
from Quantapp.visualization import Plotter
from Quantapp.analytics import Rolling
from Quantapp.data import MacroDataClient
from Quantapp.data import CompanyDataClient
from Quantapp.data import MarketDataClient
from Quantapp.visualization import PieChartPlotter
from Quantapp.visualization import BarChartPlotter
from Quantapp.visualization import LineChartPlotter
from quickfs import QuickFS
from plotly.subplots import make_subplots
from quickfs import QuickFS
import numpy as np
import json
import os
import pandas as pd
import yfinance as yf
from statsmodels.tsa.stattools import coint
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
from plotly.subplots import make_subplots
from datetime import datetime
import statsmodels.api as sm
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
import plotly.graph_objects as go
import plotly.graph_objects as go
import pandas as pd
from plotly.subplots import make_subplots
save_path = r"e:\Coding Projects\Investment Analysis"
#shut down warnings
import warnings
warnings.filterwarnings("ignore")


qc = Rolling()
qp = Plotter()
qe = MacroDataClient()
market_data = MarketDataClient()

pieChartPlotter = PieChartPlotter()
barChartPlotter = BarChartPlotter()
lineChartPlotter = LineChartPlotter()
from Quantapp.secrets import load_project_env, require_secret
load_project_env()
client = QuickFS(require_secret("API_QUICKFS"))


In [ ]:
#DOUBLE CHECK HERE THAT YOUR ARE ACCOUNTING FOR ALL INDUSTRIES

In [ ]:

#parameters

import yfinance as yf
import matplotlib.pyplot as plt

# Ticker for S&P 500 Utilities Sector
sector = "Energy"
period = "10y"
interval = "1d"

data = market_data.retrieve_market_data()
stocks = data['SP500'][data['SP500']['Sector'] == sector]
#go through each row and append the market cap to a new list
market_caps = []
for index, row in stocks.iterrows():
    try:
        ticker = yf.Ticker(row['Symbol'])
        market_cap = ticker.info['marketCap']
        market_caps.append(market_cap)
    except:
        market_caps.append(np.nan)
stocks['Market Cap'] = market_caps
sub_industries = stocks['Sub-Industry'].unique().tolist()

#dictionary of tickers in each sub-industry
sub_industry_tickers_dict = {}
for sub_industry in sub_industries:
    tickers = stocks[stocks['Sub-Industry'] == sub_industry]['Symbol'].tolist()
    sub_industry_tickers_dict[sub_industry] = tickers


In [ ]:
#Market Cap Pie Chart for Utilities Sector
pieChartPlotter.plot_sector_market_cap(sector=sector)

In [ ]:
#Market Cap Bar Chart for Utilities Sector
barChartPlotter.plot_sector_market_cap(sector=sector).show()

In [ ]:
#Sector effciency for the Utilities Sector

utility_etf = "XLU"
benchmark_etf = "SPY"
etf_data = yf.Ticker(utility_etf).history(period=period, interval=interval)
benchmark_data = yf.Ticker(benchmark_etf).history(period=period, interval=interval)
time_frame_short = 21  # Short-term window in days
time_frame_mid = 50    # Mid-term window in days
time_frame_long = 200  # Long-term window in days
length_of_plots = 10  # Number of years to display in the plot





lineChartPlotter.plot_returns(
    etf_data['Close'],
    benchmark_series=benchmark_data['Close'],
    plot_type='risk_adjusted',
    title=f'{utility_etf} Returns',
    windows=[time_frame_short, time_frame_mid, time_frame_long],
    ratio_type='sharpe',
    default_years=length_of_plots
).show()




In [ ]:
company_data = CompanyDataClient('UBER', client, save_path=save_path)

In [ ]:
#display last 3 metrics in python list
display(company_data.get_metrics()[-3:])
display(company_data.get_latest_earnings_date())
display(company_data.retrieve_data(should_update=False))
usage_data = client.get_usage()['quota']
print("Data Usage:")
print(f"  Used     : {usage_data['used']}")
print(f"  Remaining: {usage_data['remaining']}")
print(f"  Resets   : {usage_data['resets']}")

In [ ]:

data = company_data.retrieve_data(data_type="quarterly", statement_type='computed', should_update=False)
data['original_filing_date'] = pd.to_datetime(data['original_filing_date'])
data =data.set_index('original_filing_date')[['market_cap']]
data.index = data.index.strftime('%Y-%m-%d')


uber_closing_prices = yf.Ticker('UBER').history(period='10y', interval='1d')['Close']
uber_closing_prices.index = uber_closing_prices.index.strftime('%Y-%m-%d')


display(data.index)
display(uber_closing_prices.index)

#combine into single dataframe
combined_df = pd.DataFrame({
    'Market Cap': data['market_cap'],
    'Closing Price': uber_closing_prices
})
combined_df 


In [ ]:
#plot the 21 day rolling sharpe ratio for each equity in the utilities sector on one chart
sharpe_ratios = []
#plot the 21 day rolling sharpe ratio for each equity in the utilities sector on one chart
sharpe_ratios = []
for sub_industry, tickers in sub_industry_tickers_dict.items():
    for ticker_str in tickers:
        try:
            ticker = yf.Ticker(ticker_str).history(period=period, interval=interval)
            #calculate sharpe ratio by hand
            returns = ticker['Close'].pct_change().dropna()
            rolling_mean = returns.rolling(window=time_frame_long).mean()
            rolling_std = returns.rolling(window=time_frame_long).std()
            rolling_sharpe = (rolling_mean - 0.02/252) / rolling_std * np.sqrt(252)
            sharpe_ratios.append((ticker_str, rolling_sharpe))
        except:
            continue
#plot all sharpe ratios on one chart
fig = go.Figure()
for ticker_str, rolling_sharpe in sharpe_ratios:
    fig.add_trace(go.Scatter(x=rolling_sharpe.index, y=rolling_sharpe, mode='lines', name=ticker_str))
fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Zero Line")
fig.update_layout(title='21-Day Rolling Sharpe Ratios for Utilities Sector Equities', xaxis_title='Date', yaxis_title='Sharpe Ratio')
fig.show()